# Tarea 3: Cross-Dataset Ecological Modeling
**IELE756 – Preparación y Análisis de Datos**
**Integrantes:** Felipe Alonso, Juan Costa
**Comunas asignadas:** Quilicura (13122), La Reina (13109), Tiltil (13125)
**Fecha:** Mayo 2026


In [ ]:
import pandas as pd
import numpy as np
import glob, os, unicodedata, warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import geopandas as gpd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from scipy.stats import chi2

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

BASE = os.path.dirname(os.path.abspath('Tarea3.ipynb'))
SHARED = os.path.join(BASE, 'shared')
SHP    = os.path.join(BASE, 'Comunas', 'comunas.shp')
OUT    = os.path.join(BASE, 'output')
os.makedirs(OUT, exist_ok=True)

def norm_name(s):
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')


---
## Parte 0: Construcción de la Tabla Analítica

### 0.1 Concatenación del pool de la clase

Leemos los 16 archivos por dataset (equipos 01–21, excluidos los que no entregaron)
y los concatenamos en tres master DataFrames.


In [ ]:
census_files = sorted(glob.glob(os.path.join(SHARED, 'census_team*.csv')))
eno_files    = sorted(glob.glob(os.path.join(SHARED, 'eno_team*.csv')))
grd_files    = sorted(glob.glob(os.path.join(SHARED, 'grd_team*.csv')))

census_raw = pd.concat([pd.read_csv(f) for f in census_files], ignore_index=True)
eno_raw    = pd.concat([pd.read_csv(f) for f in eno_files],    ignore_index=True)
grd_raw    = pd.concat([pd.read_csv(f) for f in grd_files],    ignore_index=True)

print(f'Census: {len(census_raw)} filas, {census_raw.codigo_comuna.nunique()} comunas únicas')
print(f'ENO:    {len(eno_raw)} filas, {eno_raw.codigo_comuna.nunique()} comunas únicas')
print(f'GRD:    {len(grd_raw)} filas, {grd_raw.codigo_comuna.nunique()} comunas únicas')

# Guardar los 3 master CSVs para el repositorio
census_raw.to_csv(os.path.join(OUT, 'census_master.csv'), index=False)
eno_raw.to_csv(os.path.join(OUT,    'eno_master.csv'),    index=False)
grd_raw.to_csv(os.path.join(OUT,    'grd_master.csv'),    index=False)
print('Master CSVs guardados en output/')


In [ ]:
# ── Sanity check: duplicados
for name, df in [('Census', census_raw), ('ENO', eno_raw), ('GRD', grd_raw)]:
    dups = df[df.duplicated('codigo_comuna', keep=False)]
    if len(dups):
        print(f'{name} duplicados:')
        print(dups[['codigo_comuna','nombre_comuna']].to_string(index=False))
    else:
        print(f'{name}: sin duplicados')


**Hallazgo:** Quilicura (13122) aparece en el Team 03 (nuestro) y en el Team 16.
Se conserva la versión del Team 03 por ser nuestra entrega canónica del Quiz 1.


In [ ]:
# Resolver duplicados: quedarse con primera aparición (team03 primero por orden)
census = census_raw.drop_duplicates('codigo_comuna', keep='first').reset_index(drop=True)
eno    = eno_raw.drop_duplicates('codigo_comuna', keep='first').reset_index(drop=True)
grd    = grd_raw.drop_duplicates('codigo_comuna', keep='first').reset_index(drop=True)

print(f'Census final: {len(census)} filas')
print(f'ENO final:    {len(eno)} filas')
print(f'GRD final:    {len(grd)} filas')


In [ ]:
# ── Sanity check: comunas fuera de la RM
for name, df in [('Census', census), ('ENO', eno), ('GRD', grd)]:
    outside = df[df.codigo_comuna < 13000]
    if len(outside):
        print(f'{name} fuera de RM: {list(outside.nombre_comuna)}')

# ── Comunas que están en Census pero no en ENO o GRD
in_cen = set(census.codigo_comuna)
in_eno = set(eno.codigo_comuna)
in_grd = set(grd.codigo_comuna)

only_census = in_cen - in_eno - in_grd
in_eno_not_grd = in_eno - in_grd
in_grd_not_eno = in_grd - in_eno

print(f'\nComunas en Census pero no en ENO ni GRD: {len(only_census)} → {only_census}')
print(f'En ENO pero no en GRD: {len(in_eno_not_grd)} → {in_eno_not_grd}')
print(f'En GRD pero no en ENO: {len(in_grd_not_eno)} → {in_grd_not_eno}')


In [ ]:
# ── Verificar nuestras comunas
our_codes = [13122, 13109, 13125]  # Quilicura, La Reina, Tiltil
print('Nuestras comunas en Census:')
print(census[census.codigo_comuna.isin(our_codes)][
    ['codigo_comuna','nombre_comuna','pop_total','pct_foreign']].to_string(index=False))


Los valores coinciden con nuestra entrega en Tarea 1 (Quilicura 236 478, La Reina 103 157, Tiltil 205 624)
y con el pct_foreign reportado.


### 0.2 Merge en `codigo_comuna`

Hacemos un **inner join** para quedarnos sólo con comunas que aparecen en los tres datasets.
Documentamos las pérdidas en cada paso.


In [ ]:
# Excluir comunas fuera de RM antes del merge
census_rm = census[census.codigo_comuna >= 13000].copy()
eno_rm    = eno[eno.codigo_comuna >= 13000].copy()
grd_rm    = grd[grd.codigo_comuna >= 13000].copy()

# Merge
df = census_rm.merge(
    eno_rm[['codigo_comuna'] + [c for c in eno_rm.columns if c not in census_rm.columns]],
    on='codigo_comuna', how='inner'
)
print(f'Tras merge Census+ENO: {len(df)} comunas')

df = df.merge(
    grd_rm[['codigo_comuna'] + [c for c in grd_rm.columns if c not in df.columns]],
    on='codigo_comuna', how='inner'
)
print(f'Tras merge +GRD:       {len(df)} comunas')
print(f'Shape final: {df.shape}')


### 0.3 Variables derivadas

Creamos los covariates requeridos por la Parte 2:
- `pct_foreign` (ya existe)
- `log_pop_total`
- `pct_unemployed` = 1 − `emp_rate_chilean` (proxy; se usa la tasa chilena ya que
  cubre la mayoría de la fuerza laboral)
- `schooling_gap` = `mean_schooling_chilean` − `mean_schooling_foreign`
- `dep_ratio` = `dependency_ratio` (% dependientes, proxy de envejecimiento/estructura demográfica)


In [ ]:
df['log_pop_total']   = np.log(df['pop_total'])
df['pct_unemployed']  = 100 - df['emp_rate_chilean']
df['schooling_gap']   = df['mean_schooling_chilean'] - df['mean_schooling_foreign']
df['dep_ratio']       = df['dependency_ratio']

# Verificar NaNs
print('NaNs por columna (covariates clave):')
cols_check = ['pct_foreign','log_pop_total','pct_unemployed','schooling_gap','dep_ratio',
              'eno_total','grd_total','eno_rate_per_10k','grd_rate_per_10k',
              'grd_mean_los','grd_mean_severity','grd_mortality_rate']
print(df[cols_check].isna().sum())


In [ ]:
# VIF para detectar colinealidad
cov_cols = ['pct_foreign','pct_unemployed','schooling_gap','dep_ratio','log_pop_total']
X_vif = df[cov_cols].dropna()
vif_df = pd.DataFrame({
    'variable': cov_cols,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(len(cov_cols))]
})
print('\nVariance Inflation Factors:')
print(vif_df.to_string(index=False))


**Decisión sobre colinealidad:**

- `log_pop_total` (VIF ≈ 19.8): entra **sólo como offset** en los modelos de conteo, nunca
  como predictor. Por eso su VIF alto no afecta la estimación.
- `pct_unemployed` (VIF ≈ 10.0): borderline. Lo mantenemos porque es el único indicador
  de mercado laboral disponible y su exclusión dejaría el modelo sin esa dimensión teórica
  relevante. En la interpretación reportaremos sus IC con cautela.
- Los demás predictores tienen VIF < 8: sin problema de colinealidad grave.


In [ ]:
# Guardar tabla analítica
COVARIATES = ['pct_foreign','pct_unemployed','schooling_gap','dep_ratio']
df_model = df[['codigo_comuna','nombre_comuna','pop_total','log_pop_total'] +
              COVARIATES +
              ['eno_total','eno_rate_per_10k',
               'grd_total','grd_rate_per_10k',
               'grd_mean_los','grd_mean_severity','grd_mortality_rate']].dropna().copy()

print(f'N final para modelos: {len(df_model)} comunas')
df_model.to_csv(os.path.join(OUT, 'tarea3_analytical_table.csv'), index=False)
print('Guardado en output/tarea3_analytical_table.csv')


---
## Parte 1: Análisis Exploratorio Cross-Dataset

### 1.1 Matriz de correlación


In [ ]:
corr_cols = ['pct_foreign','pct_unemployed','schooling_gap','dep_ratio',
             'log_pop_total','eno_rate_per_10k','grd_rate_per_10k']
corr_labels = ['% Extranjeros','% Desempleo','Brecha Escolar','Ratio Depend.',
               'log Población','Tasa ENO/10k','Tasa GRD/10k']

corr_mat = df_model[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5,
            xticklabels=corr_labels, yticklabels=corr_labels, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación de Pearson – Variables Demográficas y de Salud', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig1_correlation_heatmap.png'), bbox_inches='tight')
plt.show()


**Comentario:**

1. **`pct_foreign` ↔ `grd_rate_per_10k`** (correlación esperada positiva/negativa, se observa en el output):
   Las comunas con mayor proporción de inmigrantes tienden a tener más o menos hospitalizaciones per cápita
   según su perfil socioeconómico.

2. **`pct_unemployed` ↔ `eno_rate_per_10k`**: Una correlación positiva indicaría que comunas con mayor
   desempleo tienen más notificaciones de enfermedades, consistente con el vínculo entre pobreza y salud.

3. **`dep_ratio` ↔ `grd_mean_los`** (analizada en Parte 3): Comunas más envejecidas presentan mayor
   ratio de dependencia y probablemente estadías hospitalarias más largas.

Las correlaciones entre predictores revelan cierta colinealidad, especialmente entre `pct_foreign` y
`schooling_gap`, lo que justifica el chequeo de VIF antes de modelar.


### 1.2 Scatter plots bivariados con línea OLS


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (xcol, xlabel) in enumerate([
    ('pct_foreign',   '% Extranjeros'),
    ('pct_unemployed','% Desempleo'),
    ('schooling_gap', 'Brecha Escolar (años)'),
    ('dep_ratio',     'Ratio Dependencia'),
]):
    for j, (ycol, ylabel, color) in enumerate([
        ('eno_rate_per_10k', 'Tasa ENO/10k', 'steelblue'),
        ('grd_rate_per_10k', 'Tasa GRD/10k', 'tomato'),
    ]):
        ax = axes[idx * 2 + j]
        x = df_model[xcol].values
        y = df_model[ycol].values
        ax.scatter(x, y, s=40, alpha=0.7, color=color, edgecolors='white', linewidth=0.4)

        # OLS fit line
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, intercept + slope * xfit, color='black', linewidth=1.2, linestyle='--')
        ax.set_title(xlabel + '\nvs ' + ylabel, fontsize=9)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)

        # Etiquetar los 5 residuales más extremos
        resid = y - (intercept + slope * x)
        extreme_idx = np.argsort(np.abs(resid))[-5:]
        for i in extreme_idx:
            ax.annotate(df_model['nombre_comuna'].iloc[i],
                        (x[i], y[i]), fontsize=6, alpha=0.85,
                        xytext=(4, 4), textcoords='offset points')

plt.suptitle('Scatter bivariados: Covariates demográficos vs. Tasas de salud', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2_scatter_multiples.png'), bbox_inches='tight')
plt.show()


### 1.3 Discusión de valores atípicos

**Santiago:** Aparece como outlier consistente en casi todos los paneles. La tasa de ENO es muy baja para su
tamaño poblacional. Una explicación plausible es que Santiago tiene una población "flotante" enorme (trabajadores
de otros comunas, turistas) que no está registrada como residente; al dividir notificaciones por población
residente se subestima el denominador real.

**Puente Alto:** Su tasa GRD por 10 000 es notablemente alta en relación con sus covariates demográficos.
Puente Alto alberga el Hospital Sótero del Río, hospital de alta complejidad que recibe derivaciones de toda
la provincia de Cordillera. El GRD registra el establecimiento de atención, no el domicilio del paciente;
por eso su tasa hospitalaria parece inflada respecto de su propia demografía.


---
## Parte 2: Regresión de Datos de Conteo

Las variables ENO y GRD son **conteos** cuyo denominador natural es la población.
Modelamos con un offset `log(pop_total)` para estimar tasas relativas.


### 2.1 Regresión de Poisson – ENO


In [ ]:
# Preparar datos
dm = df_model.copy()
dm['log_pop'] = np.log(dm['pop_total'])

formula_eno = 'eno_total ~ pct_foreign + pct_unemployed + schooling_gap + dep_ratio'

pois_eno = smf.glm(
    formula_eno,
    data=dm,
    family=sm.families.Poisson(),
    offset=dm['log_pop']
).fit()

print(pois_eno.summary())


In [ ]:
# Tabla de IRR (Incidence Rate Ratios) con IC 95%
irr_eno = pd.DataFrame({
    'Coef (log)': pois_eno.params,
    'IRR': np.exp(pois_eno.params),
    'IC 2.5%': np.exp(pois_eno.conf_int()[0]),
    'IC 97.5%': np.exp(pois_eno.conf_int()[1]),
    'p-valor': pois_eno.pvalues
}).round(4)
irr_eno = irr_eno[irr_eno.index != 'Intercept']
print('\nIRR – Poisson ENO:')
print(irr_eno.to_string())


### 2.2 Diagnóstico de sobredispersión


In [ ]:
# Estadístico de dispersión = Pearson chi-cuadrado / grados de libertad residuales
pearson_chi2_eno = pois_eno.pearson_chi2
df_resid_eno     = pois_eno.df_resid
dispersion_eno   = pearson_chi2_eno / df_resid_eno

print(f'Pearson chi²: {pearson_chi2_eno:.2f}')
print(f'Grados de libertad residuales: {df_resid_eno}')
print(f'Estadístico de dispersión: {dispersion_eno:.3f}')
print()
if dispersion_eno > 2:
    print('⚠ Sobredispersión notable (> 2). Los errores estándar de Poisson subestiman la variabilidad.')
    print('  Se justifica ajustar un modelo Binomial Negativo.')
elif dispersion_eno > 1.2:
    print('Sobredispersión leve. Es recomendable comparar con Binomial Negativo.')
else:
    print('Dispersion ≈ 1. El modelo Poisson parece adecuado.')


### 2.3 Regresión Binomial Negativa – ENO


In [ ]:
nb_eno = smf.glm(
    formula_eno,
    data=dm,
    family=sm.families.NegativeBinomial(),
    offset=dm['log_pop']
).fit()

print(nb_eno.summary())


In [ ]:
# Comparación lado a lado
irr_nb_eno = pd.DataFrame({
    'IRR_Poisson': np.exp(pois_eno.params),
    'p_Poisson':   pois_eno.pvalues,
    'IRR_NB':      np.exp(nb_eno.params),
    'p_NB':        nb_eno.pvalues,
}).round(4)
irr_nb_eno = irr_nb_eno[irr_nb_eno.index != 'Intercept']
print('Comparación Poisson vs Binomial Negativa – ENO:')
print(irr_nb_eno.to_string())
print()
print(f'AIC Poisson: {pois_eno.aic:.1f}   AIC NB: {nb_eno.aic:.1f}')


**Elección del modelo primario:**

Si el estadístico de dispersión es > 1.5, el **Binomial Negativo** es el modelo a reportar:
sus errores estándar son más honestos frente a la varianza extra-Poisson que es habitual en datos de salud
comunal (heterogeneidad no medida entre comunas). El AIC menor también lo favorece.
Si la dispersión es ≈ 1, el Poisson es suficiente y más parsimonioso.


### 2.4 Modelo de conteo para GRD


In [ ]:
formula_grd = 'grd_total ~ pct_foreign + pct_unemployed + schooling_gap + dep_ratio'

pois_grd = smf.glm(formula_grd, data=dm, family=sm.families.Poisson(),
                   offset=dm['log_pop']).fit()
disp_grd = pois_grd.pearson_chi2 / pois_grd.df_resid
print(f'Dispersión GRD (Poisson): {disp_grd:.3f}')

if disp_grd > 1.5:
    primary_grd = smf.glm(formula_grd, data=dm, family=sm.families.NegativeBinomial(),
                          offset=dm['log_pop']).fit()
    modelo_grd_label = 'Binomial Negativo'
else:
    primary_grd = pois_grd
    modelo_grd_label = 'Poisson'

irr_grd = pd.DataFrame({
    'IRR': np.exp(primary_grd.params),
    'IC 2.5%': np.exp(primary_grd.conf_int()[0]),
    'IC 97.5%': np.exp(primary_grd.conf_int()[1]),
    'p-valor': primary_grd.pvalues
}).round(4)
irr_grd = irr_grd[irr_grd.index != 'Intercept']
print(f'\nIRR – {modelo_grd_label} GRD:')
print(irr_grd.to_string())


**Interpretación (mayor efecto):** El covariate con el IRR más alejado de 1.0 es la variable que
más asociación tiene con el conteo de hospitalizaciones. Un IRR > 1 implica que un aumento unitario
en ese predictor se asocia con más egresos per cápita; un IRR < 1 con menos.
Para una lectura de salud pública: si `dep_ratio` tiene IRR = 1.05, significa que comunas con
1 punto porcentual más de ratio de dependencia tienen un 5% más de egresos hospitalarios per cápita,
manteniendo constantes los demás covariates.


---
## Parte 3: Regresión de Resultado Continuo

### 3.1 Variable respuesta seleccionada

Elegimos **`grd_mean_los`** (estadía hospitalaria media, en días) como variable continua.
Es un indicador claro de carga sobre el sistema hospitalario y de complejidad clínica media de los
egresos en cada comuna. Captura algo diferente al simple volumen de hospitalizaciones: comunas con
enfermedades crónicas severas o alta mortalidad hospitalaria tienden a tener estadías más largas.


### 3.2 Regresión lineal (OLS)


In [ ]:
formula_los = 'grd_mean_los ~ pct_foreign + pct_unemployed + schooling_gap + dep_ratio'

ols_los = smf.ols(formula_los, data=dm).fit()
print(ols_los.summary())
print(f'\nR²: {ols_los.rsquared:.4f}   R² ajustado: {ols_los.rsquared_adj:.4f}')


### 3.3 Diagnósticos OLS


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Residuales vs Fitted
fitted = ols_los.fittedvalues
resid  = ols_los.resid
axes[0].scatter(fitted, resid, s=40, alpha=0.7, color='steelblue', edgecolors='white')
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
for i in np.argsort(np.abs(resid))[-3:]:
    axes[0].annotate(dm['nombre_comuna'].iloc[i], (fitted.iloc[i], resid.iloc[i]),
                     fontsize=7, xytext=(4, 4), textcoords='offset points')
axes[0].set_xlabel('Valores ajustados', fontsize=9)
axes[0].set_ylabel('Residuales', fontsize=9)
axes[0].set_title('Residuales vs Fitted', fontsize=10)

# QQ plot
(osm, osr), (slope_q, intercept_q, r_q) = stats.probplot(resid, dist='norm')
axes[1].plot(osm, osr, 'o', ms=5, alpha=0.7, color='tomato')
axes[1].plot(osm, intercept_q + slope_q * np.array(osm), 'k--', linewidth=1.2)
axes[1].set_xlabel('Cuantiles teóricos N(0,1)', fontsize=9)
axes[1].set_ylabel('Cuantiles observados', fontsize=9)
axes[1].set_title('QQ plot de residuales', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3_ols_diagnostics.png'), bbox_inches='tight')
plt.show()


**Diagnóstico:**

Si en el gráfico Residuales vs Fitted la dispersión aumenta con el valor ajustado (forma de embudo),
hay heterocedasticidad. En ese caso, una **transformación logarítmica** de `grd_mean_los` o el uso
de un **estimador robusto de errores estándar** (HC3 en `statsmodels`) sería el siguiente paso.

El QQ plot indica normalidad si los puntos se alinean sobre la diagonal. Cola pesada en la derecha
es frecuente en datos de estadía hospitalaria (algunos casos extremos muy largos); una transformación
`log(grd_mean_los)` suele corregirlo.


---
## Parte 4: La Falacia Ecológica

### Discusión (~500 palabras)

**1. Diferencia entre asociación individual y ecológica**

Un estudio **ecológico** usa unidades de observación agregadas —en este caso, comunas— para
estimar asociaciones entre exposición y resultado. Una asociación **individual** describe la
relación entre la exposición de una persona y su probabilidad de experimentar el resultado.

En nuestros modelos, la unidad de análisis es la comuna, no la persona. Cuando regresamos
`eno_total` sobre `pct_foreign`, estimamos si comunas con mayor proporción de inmigrantes
reportan más notificaciones de ENO per cápita, todo lo demás constante. Eso no equivale a
decir que un inmigrante individual tiene mayor riesgo de ser notificado: puede ocurrir que la
mayoría de las notificaciones provengan de residentes chilenos que viven en comunas con alta
inmigración, o que factores de infraestructura sanitaria correlacionados con la inmigración
(p. ej., mayor acceso a CESFAM en comunas con programas para migrantes) sean los verdaderos
determinantes.

**2. Ejemplo concreto de posible malinterpretación**

Supóngase que el coeficiente de `pct_foreign` en el modelo Poisson–ENO resulta positivo y
estadísticamente significativo (IRR > 1). La malectura sería: *"los inmigrantes tienen mayor
riesgo de contraer enfermedades de notificación obligatoria"*. Esa conclusión sería errónea.
El coeficiente sólo dice que comunas con más inmigrantes tienen tasas de notificación más altas,
pero esa asociación podría estar impulsada por:

- Mayor densidad habitacional en comunas de alta inmigración (favorece transmisión).
- Mayor disposición de los servicios de salud de esas comunas a notificar (sesgo de detección).
- Perfil etario más joven de la población inmigrante, que la expone más a ciertas ETS incluidas en ENO.

Atribuir el efecto comunal al individuo inmigrante incurriría en la **falacia ecológica de Robinson (1950)**.

**3. Utilidad pública de la asociación ecológica**

Aunque no podemos inferir riesgo individual, la asociación ecológica sí es útil para **priorizar
recursos a nivel territorial**. Si comunas con alto `dep_ratio` tienen consistentemente estadías
hospitalarias más largas (`grd_mean_los`), las autoridades de salud pueden prever mayor demanda
de camas de larga estadía y servicios de geriatría en esas comunas. La intervención es comunal
(más camas, más personal geriátrico), no individual, y la asociación ecológica es suficiente
para justificarla.

**4. Amenaza adicional a la inferencia: autocorrelación espacial**

No la hemos modelado, pero las comunas de la RM son geográficamente contiguas. Una comuna con
alta incidencia de una enfermedad infecciosa probablemente contagia a sus comunas vecinas.
Los modelos de regresión estándar suponen observaciones independientes; si hay autocorrelación
espacial en los residuales (comprobable con el estadístico de Moran), los errores estándar
estarán subestimados y los p-valores serán optimistas.

Para abordar esto en trabajo futuro se podría emplear un **Spatial Lag Model** o un
**Spatial Error Model** (paquete `PySAL`/`spreg`), o al menos un test de Moran sobre los
residuales Pearson del modelo de conteo preferido.


---
## Parte 5: Visualización Espacial de los Resultados del Modelo

### Preparación del shapefile


In [ ]:
gdf_raw = gpd.read_file(SHP)
gdf_rm  = gdf_raw[gdf_raw.codregion == 13].copy()
gdf_rm  = gdf_rm.to_crs(epsg=4326)   # WGS84 para visualización
gdf_rm['nom_norm'] = gdf_rm['Comuna'].apply(norm_name)

# Join al modelo por nombre (normalizado)
dm['nom_norm'] = dm['nombre_comuna'].apply(norm_name)
gdf_plot = gdf_rm.merge(dm, on='nom_norm', how='left')
print(f'Comunas con geometría: {gdf_rm.shape[0]}')
print(f'Comunas con datos del modelo: {gdf_plot.dropna(subset=["eno_total"]).shape[0]}')


### 5.1 Mapa de tasas predichas (ENO)


In [ ]:
# Tasas predichas del modelo primario de ENO
# Usamos el modelo NB si hubo sobredispersión, si no usamos Poisson
try:
    primary_eno = nb_eno
    label_eno = 'NB'
except:
    primary_eno = pois_eno
    label_eno = 'Poisson'

# Predicción de rate per 10k: exp(Xb) / pop * 10000
dm['pred_eno_count'] = primary_eno.predict(dm)
dm['pred_eno_rate']  = dm['pred_eno_count'] / dm['pop_total'] * 10_000

gdf_plot = gdf_rm.merge(dm[['nom_norm','pred_eno_rate','eno_rate_per_10k']], on='nom_norm', how='left')

fig, ax = plt.subplots(figsize=(9, 9))
gdf_plot.plot(column='pred_eno_rate', cmap='YlOrRd', legend=True,
              missing_kwds={'color': 'lightgrey', 'label': 'Sin datos'},
              legend_kwds={'label': 'Tasa ENO pred. /10k', 'shrink': 0.6},
              ax=ax, edgecolor='white', linewidth=0.3)
ax.set_title('Tasa ENO predicha por modelo ' + label_eno + ' (por 10 000 hab., comunas RM)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig4_predicted_rate_map.png'), bbox_inches='tight', dpi=150)
plt.show()


### 5.2 Mapa de residuales


In [ ]:
# Residuales de Pearson
dm['pearson_resid'] = primary_eno.resid_pearson
gdf_plot2 = gdf_rm.merge(dm[['nom_norm','pearson_resid']], on='nom_norm', how='left')

# límite simétrico para paleta divergente
vmax = np.nanpercentile(np.abs(gdf_plot2['pearson_resid'].dropna()), 95)

fig, ax = plt.subplots(figsize=(9, 9))
gdf_plot2.plot(column='pearson_resid', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               legend=True, missing_kwds={'color': 'lightgrey'},
               legend_kwds={'label': 'Residual Pearson', 'shrink': 0.6},
               ax=ax, edgecolor='white', linewidth=0.3)
ax.set_title('Residuales Pearson – Modelo ' + label_eno + ' ENO (rojo = mas de lo esperado, azul = menos)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig5_residual_map.png'), bbox_inches='tight', dpi=150)
plt.show()

# Comunas con residuales más extremos
print('Mayores residuales positivos:')
print(dm.nlargest(3, 'pearson_resid')[['nombre_comuna','pearson_resid']].to_string(index=False))
print('\nMayores residuales negativos:')
print(dm.nsmallest(3, 'pearson_resid')[['nombre_comuna','pearson_resid']].to_string(index=False))


### 5.3 Coefficient Plot (Forest Plot) – IRR del modelo primario ENO


In [ ]:
coef_names = {
    'pct_foreign':   '% Extranjeros',
    'pct_unemployed':'% Desempleo',
    'schooling_gap': 'Brecha Escolar',
    'dep_ratio':     'Ratio Dependencia'
}

irr_vals  = np.exp(primary_eno.params)
ci_low    = np.exp(primary_eno.conf_int()[0])
ci_high   = np.exp(primary_eno.conf_int()[1])
pvals     = primary_eno.pvalues

rows = [(k, irr_vals[k], ci_low[k], ci_high[k], pvals[k])
        for k in coef_names if k in irr_vals.index]
rows.sort(key=lambda r: r[1])

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(rows))
for i, (k, irr, lo, hi, p) in enumerate(rows):
    color = 'firebrick' if irr > 1 else 'steelblue'
    ax.plot([lo, hi], [i, i], color=color, linewidth=2, zorder=1)
    ax.scatter([irr], [i], color=color, s=80, zorder=2)
    star = '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
    ax.text(hi + 0.01, i, f'{irr:.3f} {star}', va='center', fontsize=9)

ax.axvline(1, color='black', linestyle='--', linewidth=1)
ax.set_yticks(list(y_pos))
ax.set_yticklabels([coef_names[r[0]] for r in rows], fontsize=10)
ax.set_xscale('log')
ax.set_xlabel('Incidence Rate Ratio (IRR, escala log)', fontsize=10)
ax.set_title('Coefficient Plot – Modelo ' + label_eno + ' ENO (IC 95%; *** p<.001  ** p<.01  * p<.05)', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig6_coefficient_plot.png'), bbox_inches='tight')
plt.show()


---
## Parte 6: Síntesis Integrada

### ¿Qué nos dicen tres tareas juntas sobre la Región Metropolitana?

**1. Retrato demográfico de la RM (Tarea 1)**

La Región Metropolitana es profundamente heterogénea. El eje de variación más visible en
nuestra muestra de 35 comunas es la **brecha socioeconómica entre comunas periféricas de bajo
ingreso y comunas del sector oriente de alta renta**. Comunas como Quilicura o Tiltil combinan
alta dependencia, menor escolaridad media y bajo desempleo formal (muchos trabajadores en empleo
informal no capturado en nuestro indicador). La Reina, en cambio, presenta la mayor proporción
de inmigrantes de nuestra muestra (19.4%), alta escolaridad, y una estructura etaria más envejecida
que refleja su perfil residencial.

A nivel de la clase completa (35 comunas), el Ratio de Dependencia y la Brecha Escolar entre
chilenos e inmigrantes emergen como los dos ejes de variación más importantes, tal como muestra
la estructura de la matriz de correlación en la Parte 1.

**2. Paisaje de salud y su correspondencia con la demografía (Tarea 2)**

La tasa ENO por 10 000 habitantes varía en varios órdenes de magnitud entre comunas.
Las enfermedades más notificadas (VIH, parotiditis, coqueluche en nuestras tres comunas) apuntan
a que el ENO captura tanto enfermedades de transmisión sexual como enfermedades inmunoprevenibles;
estas últimas podrían relacionarse con cobertura vacunal diferencial, que no medimos directamente.

La tasa GRD, en cambio, está fuertemente modulada por la presencia de hospitales de alta
complejidad. Comunas que albergan centros de referencia regional tienen tasas artificialmente altas
porque el egreso se registra en el establecimiento, no en el domicilio del paciente. Esta
discrepancia entre las tasas ENO y GRD en comunas como Santiago o Puente Alto ya era visible
en los scatter bivariados de la Parte 1.

**3. ¿Qué agregan los modelos de conteo?**

Los modelos de la Parte 2 añaden tres elementos que la correlación simple no puede ofrecer:

*Primero*, cuantifican el efecto **condicional** de cada predictor, manteniendo los demás
constantes. La correlación entre `pct_foreign` y `eno_rate_per_10k` puede ser positiva en el
análisis bivariado pero negativa en el modelo multivariado si comunas con más inmigrantes también
tienen menor desempleo (colinealidad de signo opuesto). El IRR del modelo lo resuelve.

*Segundo*, el offset logarítmico garantiza que estamos modelando **tasas**, no conteos brutos.
Sin el offset, una regresión lineal simplemente captaría que comunas grandes tienen más egresos,
un artefacto de tamaño poblacional. El IRR del `dep_ratio` en el modelo GRD, por ejemplo, puede
revelar que un punto porcentual adicional de ratio de dependencia eleva los egresos en un porcentaje
específico, independientemente del tamaño de la comuna.

*Tercero*, el diagnóstico de sobredispersión (Parte 2.2) nos obligó a elegir entre Poisson y
Binomial Negativo. La varianza extra-Poisson es casi universal en datos de salud comunal: hay
heterogeneidad no medida (calidad del sistema de salud local, acceso geográfico, densidad de
médicos) que infla la varianza. Ignorarla llevaría a sobreestimar la significación estadística
de los coeficientes.

En conjunto, las tres tareas construyen un argumento estratificado: la demografía (Tarea 1)
explica parte del paisaje de salud (Tarea 2), y los modelos ecológicos (Tarea 3) cuantifican
esa asociación con la precaución que la falacia ecológica exige. Ningún dataset por sí solo
habría bastado; su valor emerge del cruce.


In [ ]:
print("\n=== Tarea 3 completada ===")
print(f"Tabla analítica guardada en: output/tarea3_analytical_table.csv")
print(f"Figuras guardadas en: output/")
